rnn的cell是
h=tanh(w1x+w2h+b)

In [ ]:
import torch
import torch.nn as nn


class RnnCell(nn.Module):
    def __init__(self, input_size, hidden_size) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.w1 = nn.Linear(input_size, hidden_size)
        self.w2 = nn.Linear(hidden_size, hidden_size)
        
    # input.shape(batch, seq_len, input_size) 这里希望所有batch的seq_len是一样长的
    # input_size就是embedding的长度
    def forward(self, input, hidden=None):
        batch_size, seq_len, _ = input.shape
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size, device=input.device)
        
        outputs = []

        for t in range(seq_len):
            # 第t时刻的shape(batch, input_size)
            x_t = input[:, t, :]
            # hidden.shape(batch, hidden_size)
            hidden = torch.tanh(self.w1(x_t) + self.w2(hidden))
            outputs.append(hidden)

        # torch.stack(outputs, dim=1)的shape是(batch, seq_len, out_size)
        return torch.stack(outputs, dim=1), hidden

gru cell
z=sigmoid(w_zx x+w_zh h + bias) # 控制旧 h 与候选 h~ 的混合比例，是保留新记忆的比例
r=sigmoid(w_rx x + w_rh h + bais)  # 控制计算候选 h~ 时，旧 h 参与多少（r≈0 时相当于忽略历史，只看当前 x）。
h~  = tanh(w_hx x + w_hrh(r * h)) #候选状态
final_hidden = z*h~+ (1-z)*h


In [1]:
import torch
import torch.nn as nn


class GruCell(nn.Module):
    def __init__(self, input_size, hidden_size, update_bias_init=-3.0) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.w_zx = nn.Linear(input_size, hidden_size, bias=True)
        self.w_zh = nn.Linear(hidden_size, hidden_size, bias=False)

        self.w_rx = nn.Linear(input_size, hidden_size, bias=True)
        self.w_rh = nn.Linear(hidden_size, hidden_size, bias=False)

        self.w_hx = nn.Linear(input_size, hidden_size, bias=True)
        self.w_hrh = nn.Linear(hidden_size, hidden_size, bias=False)

        # 关键：把 update gate 的 bias 初始化成负值
        nn.init.constant_(self.w_zx.bias, update_bias_init)

    # input.shape(batch, seq_len, input_size) 这里希望所有batch的seq_len是一样长的
    # input_size就是embedding的长度
    def forward(self, input, hidden=None):
        batch_size, seq_len, _ = input.shape
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size, device=input.device)
        
        outputs = []

        for t in range(seq_len):
            # 第t时刻的shape(batch, input_size)
            x_t = input[:, t, :]
            # hidden.shape(batch, hidden_size)
            # z.shape(batch,hidden_size)每个元素收到了sigmoid处理
            z = torch.sigmoid( self.w_zx(x_t)+self.w_zh(hidden))
            r = torch.sigmoid( self.w_rx(x_t)+self.w_rh(hidden))
            h_ = torch.tanh(self.w_hx(x_t) + self.w_hrh(r*hidden))#r*hidden是元素相乘
            #z*hidden 和 (1-z)h_都是元素相乘
            hidden = z*h_ + (1-z)*hidden

            outputs.append(hidden)

        # torch.stack(outputs, dim=1)的shape是(batch, seq_len, out_size)
        return torch.stack(outputs, dim=1), hidden

In [3]:
cell = GruCell(input_size=8, hidden_size=16)
x = torch.randn(4, 10, 8)
out, h = cell(x)
print(out.shape, h.shape)   # torch.Size([4, 10, 16]) torch.Size([4, 16])

# 检查初始 z 是否偏大（应接近 0.88）
with torch.no_grad():
    h0 = torch.zeros(4, 16)
    z0 = torch.sigmoid(cell.w_zx(x[:, 0, :]) + cell.w_zh(h0))
    print(z0.mean().item())  # 应该 ≈ 0.88 左右

torch.Size([4, 10, 16]) torch.Size([4, 16])
0.8611382842063904
